In [ ]:
#instalando a biblioteca Llama index
!pip install -q llama-index

In [4]:
# biblioteca para permitir a leitura de arquivos
from llama_index.core import SimpleDirectoryReader

In [5]:
documentos = SimpleDirectoryReader(input_dir='documentos')

In [ ]:
documentos.input_files

In [7]:
docs = documentos.load_data()

In [ ]:
docs

In [ ]:
len(docs)

In [ ]:
print(docs[0].get_metadata_str())

In [ ]:
docs[0].__dict__

In [12]:
# importando a biblioteca
from llama_index.core.node_parser import SentenceSplitter
node_parser= SentenceSplitter(chunk_size=1200)

In [ ]:
nodes = node_parser.get_nodes_from_documents(docs,show_progress=True)

In [ ]:
nodes

In [ ]:
len(nodes)

In [ ]:
nodes[0]

In [ ]:
nodes[9]

In [ ]:
#instalaçao do hugging face
!pip install -q llama-index-embeddings-huggingface

In [19]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [20]:
# Classe personalizada para integrar ao Chroma DB

class ChromaEmbeddingWrapper:
  def __init__(self,model_name:str): # inicializa com o modelo embedding
    self.model = HuggingFaceEmbedding(model_name = model_name)


  def __call__(self,input): # Converte a entrada para o formato compativel com o HuggingFace
    return self.model.embed(input)




In [21]:
embed_model_chroma= ChromaEmbeddingWrapper(model_name='intfloat/multilingual-e5-large')

In [ ]:
#instalaçao do chroma db
!pip install -q llama-index-vector-stores-chroma

In [23]:
#importando chroma db

import chromadb
db = chromadb.PersistentClient(path='./chroma_db')
chroma_client = db
collection_name = 'documentos_serenatto'

try:
 chroma_collection = chroma_client.get_or_create_collection(
        name=collection_name,
        embedding_function=embed_model_chroma)

except Exception as e :
  print(f'Erro ao criar coleçao {e}')


In [24]:
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

In [25]:
#criando a coleçao do banco de dados
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [26]:
embed_model = HuggingFaceEmbedding(model_name='intfloat/multilingual-e5-large')

In [27]:
from llama_index.core import VectorStoreIndex

In [28]:
index = VectorStoreIndex(nodes, storage_context=storage_context,embed_model = embed_model)

In [29]:
from llama_index.core import load_index_from_storage
index = load_index_from_storage(storage_context,embed_model = embed_model)

In [ ]:
from google.colab import userdata
Groq_api =userdata.get('api_groq_chat')

In [ ]:
!pip install -q llama-index-llms-groq

In [32]:
from llama_index.llms.groq import Groq

In [61]:
load_dotenv()
chave_api = os.getenv("API_GROQ")
llms = Groq(model = 'llama-3.3-70b-versatile',api_key=chave_api )

In [ ]:
chave_api
print(chave_api)

In [ ]:
!pip install python-dotenv

In [59]:
from dotenv import load_dotenv
import os


In [35]:
query_engine = index.as_query_engine(llm=llms,similarity_top_k=2)

In [ ]:
query_engine.query('Quais graos estao disponiveis').response

In [37]:
query_embedding = embed_model.get_text_embedding('Quais graos estao disponiveis ?')

In [ ]:
chroma_collection.query(query_embedding,n_results=2,include=['distances','embeddings'])

In [39]:
chat_engine = index.as_chat_engine(mode='context',llm = llms)

In [ ]:
pergunta = chat_engine.chat('Quais graos estao disponiveis ?').response
print(pergunta)

In [ ]:
pergunta = chat_engine.chat('Quais sao os preços dos graos ?').response
print(pergunta)

In [ ]:
pergunta = chat_engine.chat('Voce pode me dar mais detalhes sobre o Catuai amarelo ?').response
print(pergunta)

In [ ]:
pergunta = chat_engine.chat('Qual o preço dele ?').response
print(pergunta)

In [ ]:
chat_engine.chat_history

In [45]:
from llama_index.core.memory import ChatSummaryMemoryBuffer

In [46]:
memory = ChatSummaryMemoryBuffer(llm=llms,token_limit=256)

In [47]:
chat_engine= index.as_chat_engine(
    chat_mode = 'context',
    llm=llms,
    memory=memory,
    system_prompt=('''Voce é especialista em cafes da loja Serenato, uma loja online que vende graos de cafe
   torrados, sua funçao é tirar duvidas de forma simpatica e natural sobre os graos disponiveis''')
)

In [ ]:
response = chat_engine.chat('Ola')
print(response)

In [ ]:
response = chat_engine.chat('Voce pode me dar mais detalhes sobre o catui amarelo')
print(response)

In [ ]:
response = chat_engine.chat('qual o preço dele')
print(response)

In [ ]:
memory.get()

In [52]:
#resetando o chat
chat_engine.reset()

In [ ]:
response = chat_engine.chat('qual o preço dele')
print(response)

In [ ]:
# implementando a interface com o Gradio
!pip install gradio

In [55]:
import gradio as gr

In [57]:
# Criando a função para conversar com o chatbot
def converse_com_bot(message, chat_history):
    response = chat_engine.chat(message)

    if chat_history is None:
        chat_history = []

    chat_history.append({"role": "user", "content": message})
    chat_history.append({"role": "assistant", "content": response.response})

    return "", chat_history

# Criando a função para resetar o chatbot
def resetar_chat():
    chat_engine.reset()
    return []




In [ ]:
with gr.Blocks() as app:

    gr.Markdown('# Chatbot da Serenatto')
    chatbot = gr.Chatbot(type='messages')
    msg = gr.Textbox(label='Digite a sua mensagem')
    limpar = gr.Button('Limpar')

    msg.submit(converse_com_bot,[msg,chatbot],[msg,chatbot])
    limpar.click(resetar_chat,None,chatbot,queue=False)

    app.launch(debug=True)